In [3]:
"""
Red neuronal end-to-end en PyTorch para datos tabulares mixtos
Dataset: Adult Income (predice si una persona gana >50k$)
Columnas numéricas + categóricas con alta cardinalidad
"""

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

In [4]:
# ─────────────────────────────────────────────
# 1. CARGA Y PREPROCESADO DEL DATASET
# ─────────────────────────────────────────────

data = fetch_openml(name="adult", version=2, as_frame=True)
df = data.frame.copy()

# Separar target
df["target"] = (df["class"] == ">50K").astype(int)
df = df.drop(columns=["class"])

# Definir columnas por tipo
CAT_COLS = [
    "workclass",        # 8 categorías
    "education",        # 16 categorías  ← alta cardinalidad
    "marital-status",   # 7 categorías
    "occupation",       # 14 categorías  ← alta cardinalidad
    "relationship",     # 6 categorías
    "race",             # 5 categorías
    "sex",              # 2 categorías
    "native-country",   # 41 categorías  ← ALTA cardinalidad
]

NUM_COLS = [
    "age", "fnlwgt", "education-num",
    "capital-gain", "capital-loss", "hours-per-week"
]

# Limpiar valores nulos
# fetch_openml devuelve las columnas como tipo Categorical, así que hay que
# añadir "Unknown" como categoría válida antes de poder usarlo como fill value
for col in CAT_COLS:
    if hasattr(df[col], "cat"):
        df[col] = df[col].cat.add_categories("Unknown")
    df[col] = df[col].fillna("Unknown")
 
df[NUM_COLS] = df[NUM_COLS].fillna(df[NUM_COLS].median())

In [5]:
# ─────────────────────────────────────────────
# 2. ENCODING DE CATEGÓRICAS
#    Convertir cada categoría a un índice numérico
#    (la Embedding layer necesita índices, no strings)
# ─────────────────────────────────────────────

cat_vocab_sizes = {}  # {columna: nº de categorías únicas}
cat_embed_dims  = {}  # {columna: dimensión del embedding}

for col in CAT_COLS:
    df[col] = df[col].astype("category")
    n_cats = df[col].cat.categories.shape[0]
    cat_vocab_sizes[col] = n_cats
    # Regla práctica: embedding_dim = min(50, (n_cats // 2) + 1)
    cat_embed_dims[col]  = min(50, (n_cats // 2) + 1)
    df[col] = df[col].cat.codes  # 0, 1, 2, ...

print("Categorías y dimensiones de embedding:")
for col in CAT_COLS:
    print(f"  {col:20s} → {cat_vocab_sizes[col]:3d} cats → embed dim {cat_embed_dims[col]}")

Categorías y dimensiones de embedding:
  workclass            →   9 cats → embed dim 5
  education            →  17 cats → embed dim 9
  marital-status       →   8 cats → embed dim 5
  occupation           →  15 cats → embed dim 8
  relationship         →   7 cats → embed dim 4
  race                 →   6 cats → embed dim 4
  sex                  →   3 cats → embed dim 2
  native-country       →  42 cats → embed dim 22


In [6]:
# ─────────────────────────────────────────────
# 3. SPLIT Y NORMALIZACIÓN DE NUMÉRICAS
# ─────────────────────────────────────────────

X_num = df[NUM_COLS].values.astype(np.float32)
X_cat = df[CAT_COLS].values.astype(np.int64)
y     = df["target"].values.astype(np.float32)

X_num_tr, X_num_te, X_cat_tr, X_cat_te, y_tr, y_te = train_test_split(
    X_num, X_cat, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_num_tr = scaler.fit_transform(X_num_tr)  # fit solo en train
X_num_te = scaler.transform(X_num_te)

In [7]:
# ─────────────────────────────────────────────
# 4. DATASET PYTORCH
# ─────────────────────────────────────────────

class TabularDataset(Dataset):
    def __init__(self, X_num, X_cat, y):
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.y     = torch.tensor(y,     dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_num[idx], self.X_cat[idx], self.y[idx]

train_ds = TabularDataset(X_num_tr, X_cat_tr, y_tr)
test_ds  = TabularDataset(X_num_te, X_cat_te, y_te)

train_dl = DataLoader(train_ds, batch_size=512, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=512, shuffle=False)

In [8]:
# ─────────────────────────────────────────────
# 5. MODELO END-TO-END
#    - Embedding layer por cada columna categórica
#    - BatchNorm para las numéricas
#    - Concatenación → MLP → output
# ─────────────────────────────────────────────

class TabularNet(nn.Module):
    def __init__(self, cat_vocab_sizes, cat_embed_dims, n_num, hidden_dims, dropout=0.3):
        super().__init__()

        # Una Embedding layer por columna categórica
        # nn.Embedding(vocab_size, embed_dim):
        #   - vocab_size: cuántas categorías hay
        #   - embed_dim:  a cuántas dimensiones mapeamos cada una
        self.embeddings = nn.ModuleList([
            nn.Embedding(vocab_size + 1, embed_dim)  # +1 por si hay índices desconocidos
            for vocab_size, embed_dim in zip(cat_vocab_sizes, cat_embed_dims)
        ])

        total_embed_dim = sum(cat_embed_dims)

        # BatchNorm sobre las columnas numéricas
        self.num_norm = nn.BatchNorm1d(n_num)

        # Dimensión de entrada al MLP
        input_dim = n_num + total_embed_dim

        # MLP: capas fully connected con ReLU y Dropout
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers += [
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            prev_dim = hidden_dim

        # Capa de salida: 1 neurona + sigmoid → probabilidad binaria
        layers.append(nn.Linear(prev_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_num, x_cat):
        # Pasar cada columna categórica por su embedding
        # x_cat tiene shape [batch, n_cat_cols]
        embedded = [
            emb(x_cat[:, i])          # shape: [batch, embed_dim_i]
            for i, emb in enumerate(self.embeddings)
        ]
        # torch.cat para concatenar los embeddings de todas las columnas → [batch, total_embed_dim]
        x_emb = torch.cat(embedded, dim=1)  # [batch, total_embed_dim]

        # Normalizar numéricas
        x_num = self.num_norm(x_num)        # [batch, n_num]

        # Concatenar todo
        x = torch.cat([x_num, x_emb], dim=1)  # [batch, n_num + total_embed_dim]

        # Pasar por el MLP
        return self.mlp(x).squeeze(1)      # [batch]

# Instanciar el modelo
model = TabularNet(
    cat_vocab_sizes = list(cat_vocab_sizes.values()),
    cat_embed_dims  = list(cat_embed_dims.values()),
    n_num           = len(NUM_COLS),
    hidden_dims     = [256, 128, 64],
    dropout         = 0.3
)

print(f"\nParámetros totales: {sum(p.numel() for p in model.parameters()):,}")


Parámetros totales: 60,420


In [9]:
# ─────────────────────────────────────────────
# 6. ENTRENAMIENTO
# ─────────────────────────────────────────────

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()  # combina sigmoid + binary cross entropy

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for x_num, x_cat, y in loader:
        x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x_num, x_cat)
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for x_num, x_cat, y in loader:
            x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
            logits = model(x_num, x_cat)
            loss   = criterion(logits, y)
            total_loss += loss.item() * len(y)
            preds   = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == y).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n

EPOCHS = 10
print(f"\nEntrenando en {device} durante {EPOCHS} épocas...")
print(f"{'Época':>6} {'Train Loss':>12} {'Val Loss':>10} {'Val Acc':>10}")
print("─" * 44)

for epoch in range(1, EPOCHS + 1):
    train_loss          = train_epoch(model, train_dl, optimizer, criterion)
    val_loss, val_acc   = evaluate(model, test_dl, criterion)
    print(f"{epoch:>6}  {train_loss:>10.4f}  {val_loss:>10.4f}  {val_acc:>9.2%}")


Entrenando en cpu durante 10 épocas...
 Época   Train Loss   Val Loss    Val Acc
────────────────────────────────────────────
     1      0.3918      0.3371     84.24%
     2      0.3371      0.3221     84.98%
     3      0.3258      0.3165     85.43%
     4      0.3199      0.3109     85.91%
     5      0.3167      0.3113     85.87%
     6      0.3133      0.3141     85.63%
     7      0.3120      0.3119     85.88%
     8      0.3098      0.3094     85.84%
     9      0.3094      0.3104     85.90%
    10      0.3091      0.3108     85.67%


In [10]:
# ─────────────────────────────────────────────
# 7. INSPECCIONAR LOS EMBEDDINGS APRENDIDOS
#    Los embeddings de "education" reflejan
#    jerarquía educativa aprendida de los datos
# ─────────────────────────────────────────────

edu_categories = data.frame["education"].astype("category").cat.categories.tolist()
edu_emb_weight = model.embeddings[1].weight.detach().cpu().numpy()

print("\nEmbedding de 'education' (primeras 4 dims):")
for cat, vec in zip(edu_categories[:8], edu_emb_weight[:8]):
    dims = " ".join(f"{v:+.2f}" for v in vec[:4])
    print(f"  {cat:20s} → [{dims} ...]")

print("\nCategorías con vectores similares están cerca en el espacio embedding.")
print("La red ha aprendido automáticamente la estructura semántica.")


Embedding de 'education' (primeras 4 dims):
  10th                 → [-0.30 +0.07 -0.37 -0.40 ...]
  11th                 → [-1.65 -0.60 +0.96 +2.18 ...]
  12th                 → [-0.35 -0.08 -0.79 +0.46 ...]
  1st-4th              → [-0.43 -0.28 -0.94 -0.16 ...]
  5th-6th              → [+0.66 -1.41 -0.35 -0.46 ...]
  7th-8th              → [+0.03 +1.45 -1.22 +1.33 ...]
  9th                  → [-1.45 +1.74 +0.38 +1.53 ...]
  Assoc-acdm           → [+0.48 -0.27 -0.69 +0.38 ...]

Categorías con vectores similares están cerca en el espacio embedding.
La red ha aprendido automáticamente la estructura semántica.
